Test with data before building the EM, pruning, etc.

Create a simple vocabulary for training

In [2]:
vocab = {
    "play": 0.30,
    "ing": 0.20,
    "playing": 0.50,
    "pla": 0.05,
    "ying": 0.05,
    "p": 0.01,
    "l": 0.01,
    "a": 0.01,
    "y": 0.01,
    "i": 0.01,
    "n": 0.01,
    "g": 0.01,
}

Generate all valid token matches

In [3]:
def find_matches(text, vocab):
    matches = {}
    for start in range(len(text)):
        matches[start] = []
        for token in vocab:
            if text.startswith(token, start):
                matches[start].append(token)
    return matches

In [4]:
text = "playing"

matches = find_matches(text, vocab)
for pos, tokens in matches.items():
    if tokens:
        print(pos, tokens)

0 ['play', 'playing', 'pla', 'p']
1 ['l']
2 ['a']
3 ['ying', 'y']
4 ['ing', 'i']
5 ['n']
6 ['g']


Convert probabilities to scores

P(segmentation) = P(token1) * P(token2) *....

Multiplication is not ideal, we take logs:

log(P1 x P2) = log(P1) + log(P2)

In [5]:
import math

In [6]:
log_vocab = {
    token: math.log(prob)
    for token, prob in vocab.items()
}

VIterbi Dynamic programming

In [7]:
def viterbi(text, vocab):
    import math
    n = len(text)

    best_score = [-float("inf")] * (n + 1)
    best_prev = [-1] * (n + 1)
    best_token = [""] * (n + 1)
    best_score[0] = 0

    for pos in range(n):
        if best_score[pos] == -float("inf"):
            continue
        for token, prob in vocab.items():
            if text.startswith(token, pos):
                next_pos = pos + len(token)
                score = best_score[pos] + math.log(prob)
                if score > best_score[next_pos]:
                    best_score[next_pos] = score
                    best_prev[next_pos] = pos
                    best_token[next_pos] = token

    return best_score, best_prev, best_token

Recover the best path

In [8]:
def backtrack(text, best_prev, best_token):
    pos = len(text)
    tokens = []
    while pos > 0:
        tokens.append(best_token[pos])
        pos = best_prev[pos]
    return list(reversed(tokens))

In [9]:
scores, prev, tok = viterbi("playing", vocab)
tokens = backtrack("playing", prev, tok)

print(tokens)

['playing']


# Moving on to EM

In [10]:
from collections import Counter

def e_step(corpus, vocab):
    counts = Counter()
    for text in corpus:
        scores, prev, tok = viterbi(text, vocab)
        tokens = backtrack(text, prev, tok)
        counts.update(tokens)
    return counts

In [11]:
corpus = [
    "playing",
    "played",
    "player",
    "playing",
    "playing",
]

counts = e_step(corpus, vocab)
print(counts)

Counter({'playing': 3, '': 2})


Idea: If a token was used frequently, it's probability needs to increase.

P(token) = token_count/total_token_count

In [12]:
def m_step(counts):
    total = sum(counts.values())
    new_vocab = {}
    for token, count in counts.items():
        new_vocab[token] = count / total
    return new_vocab

In [13]:
new_vocab = m_step(counts)

for token, prob in sorted(
    new_vocab.items(),
    key=lambda x: x[1],
    reverse=True,
):
    print(token, round(prob, 3))

playing 0.6
 0.4


One EM Iteration

In [14]:
def em_iteration(corpus, vocab):
    counts = e_step(corpus, vocab)
    vocab = m_step(counts)
    return vocab

In [15]:
vocab = em_iteration(corpus, vocab)
print(vocab)

{'playing': 0.6, '': 0.4}


Extract *All substrings

In [16]:
from collections import Counter

def extract_substrings(word, max_len=10):
    n = len(word)
    for start in range(n):
        for end in range(start + 1,
                         min(n + 1, start + max_len + 1)):
            yield word[start:end]

In [17]:
list(extract_substrings("play", max_len=4))

['p', 'pl', 'pla', 'play', 'l', 'la', 'lay', 'a', 'ay', 'y']

Count corpus frequencies

In [18]:
def build_candidate_counts(corpus, max_len=10):
    counts = Counter()
    for word in corpus:
        counts.update(extract_substrings(word, max_len=max_len))
    return counts

In [19]:
counts = build_candidate_counts(corpus)
counts.most_common(20)

[('p', 5),
 ('pl', 5),
 ('pla', 5),
 ('play', 5),
 ('l', 5),
 ('la', 5),
 ('lay', 5),
 ('a', 5),
 ('ay', 5),
 ('y', 5),
 ('playi', 3),
 ('playin', 3),
 ('playing', 3),
 ('layi', 3),
 ('layin', 3),
 ('laying', 3),
 ('ayi', 3),
 ('ayin', 3),
 ('aying', 3),
 ('yi', 3)]

Keep only strong candidates

In [20]:
def build_seed_vocab(corpus, vocab_size=100, max_len=10):
    counts = build_candidate_counts(corpus, max_len=max_len)
    vocab = {}
    for token, freq in counts.most_common(vocab_size):
        vocab[token] = freq
    return vocab

In [21]:
seed_vocab = build_seed_vocab(corpus, vocab_size=50)
for token, freq in list(seed_vocab.items())[:20]:
    print(token, freq)

p 5
pl 5
pla 5
play 5
l 5
la 5
lay 5
a 5
ay 5
y 5
playi 3
playin 3
playing 3
layi 3
layin 3
laying 3
ayi 3
ayin 3
aying 3
yi 3


Converting frequencies to probabilities

In [22]:
def normalize_vocab(vocab):
    total = sum(vocab.values())
    return {
        token: freq / total
        for token, freq in vocab.items()
    }

In [23]:
vocab = normalize_vocab(seed_vocab)

Sort by Probability

In [24]:
def prune_vocab(vocab, target_size, atomic_tokens=None):
    if atomic_tokens is None:
        atomic_tokens = set()
    sorted_tokens = sorted(
        vocab.items(),
        key=lambda x: x[1],
        reverse=True
    )
    new_vocab = {}
    for token, prob in sorted_tokens:
        if len(new_vocab) >= target_size:
            break
        new_vocab[token] = prob
    for token in atomic_tokens:
        if token not in new_vocab:
            new_vocab[token] = 1e-12
    return new_vocab

We need to preserve the atomic symbols

In [25]:
def get_atomic_tokens(corpus):
    chars = set()
    for word in corpus:
        chars.update(word)
    return chars

In [26]:
atomic_tokens = get_atomic_tokens(corpus)
print(sorted(atomic_tokens))

['a', 'd', 'e', 'g', 'i', 'l', 'n', 'p', 'r', 'y']


Training loop

In [27]:
def train_unigram(corpus, initial_vocab_size=500, final_vocab_size=50, em_steps=2):
    atomic_tokens = get_atomic_tokens(corpus)
    seed_vocab = build_seed_vocab(
        corpus,
        vocab_size=initial_vocab_size
    )
    vocab = normalize_vocab(seed_vocab)
    while len(vocab) > final_vocab_size:
        for _ in range(em_steps):
            counts = e_step(corpus, vocab)
            total = sum(counts.values())
            vocab = {
                token: count / total
                for token, count in counts.items()
            }
        next_size = max(final_vocab_size, int(len(vocab) * 0.8))
        vocab = prune_vocab(vocab, next_size, atomic_tokens)
    return vocab

In [28]:
corpus = [
    "playing",
    "played",
    "player",
    "playful",
    "playground",
    "replaying",
    "replayed",
]

In [29]:
vocab = train_unigram(
    corpus,
    initial_vocab_size=200,
    final_vocab_size=30
)

In [30]:
for token, prob in sorted(vocab.items(), key=lambda x: x[1], reverse=True):
    print(f"{token:15s}", round(prob, 4))

playing         0.1429
played          0.1429
player          0.1429
playful         0.1429
playground      0.1429
replaying       0.1429
replayed        0.1429
n               0.0
r               0.0
l               0.0
o               0.0
y               0.0
d               0.0
p               0.0
a               0.0
f               0.0
i               0.0
u               0.0
e               0.0
g               0.0
